# Lead-lag multi-echelle a haute frequence : replication de Hayashi & Koike (2020)

**Leo Renault, Theo Verdelhan, Ziad El Arari - M2 272 Gestion Quantitative, Paris-Dauphine PSL**

Ce notebook raconte le projet de bout en bout : la question, la methode HK, l'exploration
des donnees, puis les resultats. **Le resultat central est le lead-lag cross-venue sur
le meme actif** (BTC sur Binance vs Kraken, spot vs perp), qui est le setup le plus proche
de celui du papier (NASDAQ vs BATS pour le meme titre).

Il fonctionne en mode *from artifacts* : il lit les tableaux de resultats (`outputs/*/summary.csv`)
et affiche les figures (`prez/figures/`), sans avoir besoin de recharger les donnees brutes.
Pour la replication step-by-step des briques de code, voir `01_walkthrough.ipynb`.

**Convention de signe** (verrouillee par `scripts/audit_sign_convention.py`) :
- $\hat\theta_j > 0 \Leftrightarrow$ serie 2 mene serie 1
- $\hat\theta_j < 0 \Leftrightarrow$ serie 1 mene serie 2

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()              # hk_leadlag_replication/
OUT  = ROOT / 'outputs'                   # tableaux de resultats (summary.csv, *.json)
FIGS = ROOT.parent / 'prez' / 'figures'   # figures pre-generees

def show(fig_name, width=820):
    # Affiche une figure depuis prez/figures/ (fallback outputs/)
    for base in (FIGS, OUT):
        p = base / fig_name
        if p.exists():
            display(Image(filename=str(p), width=width)); return
    print(f'[figure absente : {fig_name}]')

def summary(run):
    # Charge le summary.csv d'un run
    return pd.read_csv(OUT / run / 'summary.csv')

def stats(run):
    # Charge series_stats.json d'un run
    return json.loads((OUT / run / 'series_stats.json').read_text())

print('OK - artefacts lus depuis', OUT)
print('     figures lues depuis ', FIGS)

## 1. La question

Dans un marche haute frequence, certains actifs (ou certaines venues du meme actif)
reagissent **avant** d'autres a l'information. Quantifier ce **lead-lag** est :
- un test d'efficience faible (Fama 1970),
- un signal potentiellement exploitable (arbitrage statistique HF).

**Trois difficultes pour le mesurer en HF** :
1. **Non-synchronicite** des ticks (deux actifs ne sont pas observes aux memes instants).
2. **Bruit de microstructure** (effet Epps, Reno 2003 : la correlation s'effondre quand le pas tend vers 0).
3. **Nature multi-echelle** : le lag a 1 ms n'est pas celui a 1 s ou 1 min.

## 2. La methode de Hayashi-Koike

Trois briques imbriquees.

### Brique 1 : Hayashi-Yoshida non-synchrone (laggee)
$$\hat U_N(\tau) = \sum_{I \in \mathcal{O}^1,\, J \in \mathcal{O}^2,\, I \cap (J+\tau) \neq \emptyset} \Delta X^1(I)\, \Delta X^2(J)$$
Somme sur les paires d'intervalles qui se chevauchent : pas d'interpolation, exploite la resolution maximale.

### Brique 2 : convolution par ondelette (Daubechies db10)
$$\hat\Gamma_j(\tau) = \sum_l \hat U_N(\tau - l\,\Delta_N)\, \Psi_j(l)$$
$\Psi_j$ est l'autocorrelation de l'ondelette au niveau $j$, qui agit comme un filtre passe-bande
sur la bande dyadique d'echelle $j$. **C'est ici qu'interviennent les ondelettes : elles filtrent
la cross-covariance $\hat U_N$, pas les prix eux-memes.**

### Brique 3 : argmax par echelle
$$\hat\theta_j = \arg\max_{\tau \in G_N} |\hat\Gamma_j(\tau)|$$

**Lien avec le cours Garcin** (covariance multiresolution, slides 73-77 ; lead-lag DAX/S&P500).
HK font trois generalisations : discret vers continu, synchrone vers non-synchrone, scalaire
vers fonction du lag.

## 3. Validation : audit de la convention de signe

Audit synthetique : on construit deux series ou **serie 1 mene serie 2 de +3 s**
(serie 2 = copie retardee de serie 1). L'estimateur retourne **-3.00**, pas +3.

Donc la convention du codebase est :
- $\hat\theta_j > 0 \Leftrightarrow$ **serie 2 mene serie 1**
- $\hat\theta_j < 0 \Leftrightarrow$ **serie 1 mene serie 2**

A executer (necessite le venv) :

In [ ]:
# Audit reproductible de la convention de signe (optionnel : necessite le venv configure)
import subprocess, sys
try:
    out = subprocess.run([sys.executable, '../scripts/audit_sign_convention.py'],
                         capture_output=True, text=True, timeout=120)
    print(out.stdout[-1500:] or out.stderr[-1500:])
except Exception as e:
    print('Audit non execute ici :', e)
    print('Resultat attendu : theta_hat ~ -3.00 quand serie 1 mene serie 2 de +3 s.')

## 4. Exploration des donnees

Le coeur de notre application est le **meme actif sur deux venues** : BTC sur Binance et
sur Kraken (avril 2026, 30 jours). C'est le setup le plus proche du papier (NASDAQ vs BATS).

La premiere chose a regarder : les deux venues n'echantillonnent pas a la meme cadence.
Binance est beaucoup plus actif que Kraken, ce qui motive un estimateur robuste a la
non-synchronicite (Hayashi-Yoshida) plutot qu'un alignement naif sur grille.

In [ ]:
s = stats('2026-04_full_month_btc_binance_vs_kraken')
print('BTC, avril 2026 (30 jours)')
print(f"  Binance : {s['n1']:>12,} trades   median inter-tick = {s['median_dt1_ms']:7.1f} ms")
print(f"  Kraken  : {s['n2']:>12,} trades   median inter-tick = {s['median_dt2_ms']:7.1f} ms")
print(f"  Ratio de cadence (median) : {s['median_dt2_ms']/s['median_dt1_ms']:.0f}x")
print(f"  Duree : {s['duration_seconds']/86400:.0f} jours")

In [ ]:
# Apercu des series de prix (BTC vs ETH, ici pour illustrer la structure des donnees brutes)
show('data_overview_btc_eth.png', width=900)

## 5. Resultat central : lead-lag cross-venue sur le meme actif (BTC)

Setup le plus proche de HK 2020 : **meme actif, deux venues**. Sur 30 jours (avril 2026),
$\Delta_N = 50$ ms, $j_{max}=8$.

### 5.1 BTC : Binance vs Kraken

In [ ]:
bk = summary('2026-04_full_month_btc_binance_vs_kraken')
bk['lecture'] = bk['theta_hat_s'].apply(
    lambda t: 'Binance mene Kraken' if t < -1e-3 else ('Kraken mene Binance' if t > 1e-3 else 'synchrone'))
bk[['j','period_min_s','period_max_s','theta_hat_s','contrast_peak','lecture']]

In [ ]:
show('heatmap_btc_binance_vs_kraken_full_month.png', width=820)

**Lecture** : Binance mene Kraken de facon **monotone**, de -50 ms a l'echelle fine
jusqu'a **-1.95 s** a l'echelle grossiere ($j=8$, bande 12.8-25.6 s). Le contrast peak
grandit avec l'echelle (signal d'autant plus net que l'horizon est long).

**Important** : on observe une **dependance de la magnitude a l'echelle** (le lag apparait
et grandit aux grandes echelles), mais **pas une inversion de direction** selon l'echelle.
La bimodalite fine/grossiere que HK documentent sur NASDAQ/BATS (un venue mene aux fines
echelles, l'autre aux grossieres) n'est pas reproduite ici : notre regime temporel
($\Delta_N = 50$ ms) est bien plus grossier que le leur (~0.1 ms).

### 5.2 Generalisation : panel de 6 setups cross-venue / cross-product (30 jours)

In [ ]:
panel_runs = {
    'BTC Binance vs Kraken'        : '2026-04_full_month_btc_binance_vs_kraken',
    'ETH Binance vs Kraken'        : '2026-04_full_month_eth_binance_vs_kraken',
    'BTC USDT-M perp vs COIN-M perp': '2026-04_full_month_btc_usdt_perp_vs_coin_perp',
    'BTC spot vs USDT-M perp'      : '2026-04_full_month_btc_spot_vs_perp',
    'ETH spot vs USDT-M perp'      : '2026-04_full_month_eth_spot_vs_perp',
    'BTC Binance vs Bybit'         : '2026-04_full_month_btc_binance_vs_bybit',
}
rows = []
for label, run in panel_runs.items():
    df = summary(run)
    t8 = df['theta_hat_s'].iloc[-1]
    rows.append({'setup': label, 'theta(j=8) s': round(t8, 2),
                 'contrast(j=8)': round(df['contrast_peak'].iloc[-1], 3)})
pd.DataFrame(rows)

In [ ]:
show('comparison_panel_full_month.png', width=900)

**Trois patterns descriptifs** (calibration ~70%, hypotheses non demontrees par nos donnees) :

1. **Lag structurel cross-venue** : Binance mene Kraken (BTC et ETH), USDT-M mene COIN-M.
   Hypothese compatible : asymetrie d'infrastructure / composition des participants (HFT vs retail).
2. **Spot suit perp** : le perp mene le spot (+350 a +400 ms a $j=8$), sur BTC comme ETH.
   Compatible avec la litterature crypto (leverage + funding drivent la decouverte de prix).
3. **Cross-exchange HFT-aligne** : Binance et Bybit quasi-synchrones (un faible biais Bybit
   de +200 ms apparait seulement sur 30 jours).

Le pattern qualitatif est **stable** de 7 a 30 jours (ecarts < 20% par echelle).

### 5.3 Inference : bootstrap par blocs sur BTC Binance vs Kraken

Bootstrap par blocs mobiles (300 s), $B=50$, fenetre 24 h, correction multi-tests
Romano-Wolf (FWER) et Benjamini-Hochberg (FDR).

In [ ]:
bs = json.loads((OUT / '2026-04-13to19_btc_binance_vs_kraken' / 'bootstrap.json').read_text())
boot = pd.DataFrame({
    'j': bs['levels'],
    'theta_obs_s': np.round(bs['theta_obs'], 3),
    'CI_low_s': np.round(bs['lower'], 3),
    'CI_high_s': np.round(bs['upper'], 3),
    'p_RW': np.round(bs['romano_wolf_p'], 3),
})
display(boot)
print(f"\nFenetre : {bs['hours_cap']} h cap, B = {bs['n_ok']}/{bs['n_total']} blocs de {bs['block_length_s']} s.")
print('p_RW = 0.020 est le plancher mecanique 1/(B+1) pour B=50.')

In [ ]:
show('bootstrap_boxplot_binance_kraken.png', width=760)

**Lecture** : aux echelles $j=5..8$, le CI 95% exclut zero et $p_{RW}=0.020$ (le plus fin
que $B=50$ permet). A $j=8$, le CI est serre (~0.4 s autour de -2.1 s sur la fenetre 24 h).
C'est le resultat **statistiquement le plus solide** du projet. Caveat : $B=50$ est faible
(compute-bound) ; on ne distingue pas "tres significatif" de "moderement significatif".

## 6. Pourquoi multi-echelle ? HK vs HRY single-scale

HRY (Hoffmann-Rosenbaum-Yoshida 2013) estime **un seul** $\hat\theta$ (argmax direct de
$|\hat U_N|$, sans filtrage par ondelette). Sur BTC/ETH, il renvoie ~0 alors que HK revele
un profil structure : les contributions des echelles se compensent dans l'agregat scalaire.

In [ ]:
hk  = summary('2026-04-13to19_btc_eth_spot_full168h')['theta_hat_s'].to_numpy()
hry = summary('2026-04-13to19_btc_eth_spot_hry_baseline')['theta_hat_s'].to_numpy()
print('HK multi-echelle  theta_hat (ms):', np.round(hk * 1000).astype(int))
print('HRY single-scale  theta_hat (ms):', np.round(hry * 1000).astype(int))
print('\nLecture : HRY donne ~0 ; HK revele la structure multi-echelle.')

In [ ]:
show('comparison_hk_vs_hry.png', width=900)

## 7. BTC/ETH cross-asset : un resultat exploratoire fragile au proxy

Sur BTC vs ETH (cross-asset, pas le meme actif), on observe bien un profil multi-echelle
sur aggTrades (ETH semble mener BTC, +50 a +250 ms). **Mais la direction depend du proxy
de prix** : sur la meme semaine, trois reconstructions donnent trois directions.

In [ ]:
proxies = {
    'aggTrades (last-trade)'   : '2026-04-13to19_btc_eth_spot_full168h',
    'synth-midpoint'           : '2026-04-13to19_btc_eth_synthmid_full168h',
    'synth-microprice (Stoll)' : '2026-04-13to19_btc_eth_synthmicroprice_full168h',
}
rows = []
for label, run in proxies.items():
    t8 = summary(run)['theta_hat_s'].iloc[-1]
    direction = 'ETH mene BTC' if t8 > 1e-3 else ('BTC mene ETH' if t8 < -1e-3 else 'quasi-nul')
    rows.append({'proxy de prix': label, 'theta(j=8) s': round(t8, 3), 'direction': direction})
pd.DataFrame(rows)

In [ ]:
show('comparison_proxies_btc_eth.png', width=900)

**Conclusion defensive** : plus on s'approche d'un vrai micro-price, plus le signal cross-asset
BTC/ETH s'attenue. Une partie du $\hat\theta_j$ observe sur aggTrades est probablement un
artefact de proxy (bid-ask bounce). C'est pour cela que le coeur du projet est le **cross-venue
same-asset** (section 5), ou les trois proxies coincident en direction.

## 8. Robustesse temporelle : Calm week vs FOMC week

Re-run BTC/ETH sur la semaine FOMC (27 avril - 3 mai 2026).

In [ ]:
calm = summary('2026-04-13to19_btc_eth_spot_full168h')[['j','period_min_s','period_max_s','theta_hat_s']]
fomc = summary('2026-04-27to05-03_btc_eth_spot_fomc_week')['theta_hat_s']
cmp = calm.rename(columns={'theta_hat_s':'theta_calm_s'})
cmp['theta_fomc_s'] = fomc.to_numpy()
cmp['diff_ms'] = ((cmp['theta_fomc_s'] - cmp['theta_calm_s']) * 1000).round(0)
cmp

In [ ]:
show('comparison_calm_vs_fomc.png', width=900)

**Observation** : structure qualitative robuste, lags coarse-scale plus longs en regime FOMC
(+40% a $j=8$). La magnitude depend du regime de marche : a presenter comme dependance au
regime, pas comme instabilite methodologique.

## 9. Replication equity sur LOBSTER (sanity check quote-based)

HK utilisent des micro-prices de quotes. LOBSTER fournit des midpoints NASDAQ (un seul jour,
21 juin 2012). On ne peut pas repliquer NASDAQ vs BATS (LOBSTER = NASDAQ seul), donc on fait
un cross-asset AAPL vs MSFT : ca valide que l'estimateur tourne sur quote-midpoints.

In [ ]:
summary('2012-06-21_aapl_msft_lobster_lvl10_full')

In [ ]:
show('heatmap_aapl_msft_lobster.png', width=900)

**Observation** : signal plus bruite (1 jour), mais a $j=8$ on lit $\hat\theta_8 \approx -70$ ms,
soit AAPL menant MSFT. A presenter comme sanity check methodologique, pas comme replication
empirique du papier.

## 10. Limitations

1. **Prix** : aggTrades crypto (pas de bookTicker historique gratuit) au lieu des micro-prices de HK.
2. **Resolution** : $\Delta_N = 50$ ms (inter-tick crypto) vs ~0.1 ms TAQ chez HK.
3. **Timestamping** : timestamps exchange-API, pas de SIP consolide ; le lag mesure peut meler
   decouverte de prix et conventions de reporting.
4. **Panel** : ~10 setups cross-venue vs 108 x 21 ticker-jours chez HK ; pas de bimodalite reproduite.
5. **Inference** : bootstrap $B=50$ compute-bound, $p_{RW}$ plafonne a $1/(B+1)$.
6. **Convention de signe** : verrouillee par audit, mais piege a expliciter.

## 11. Conclusion

HK comblent l'ecart entre l'analyse multiresolution discrete (cours Garcin) et l'estimation
HF non-synchrone (Hayashi-Yoshida 2005). Nous repliquons l'estimateur et l'adaptons a des
donnees crypto/equity 2025-2026.

**Findings (calibres)** :
1. **HK revele ce que HRY masque** : sur BTC/ETH, HRY donne ~0, HK montre un profil multi-echelle.
   Argument methodologique principal. Solide.
2. **Cross-venue same-asset (le coeur)** : Binance mene Kraken de facon monotone (-50 ms a -1.95 s
   a $j=8$, 30 jours, bootstrap $p_{RW}=0.020$). Pattern confirme sur ETH ; USDT-M mene COIN-M ;
   perp mene spot ; Binance ~ Bybit. Le setup le plus proche de HK.
3. **Fragilite au proxy** : sur BTC/ETH cross-asset, trois proxies donnent trois directions.
   A presenter avec ce caveat ; le finding robuste est le cross-venue.
4. **Robustesse temporelle** : magnitude dependante du regime, pattern qualitatif stable.

**Portee** : nous repliquons la *methodologie* HK et l'adaptons a des donnees contemporaines,
pas le protocole empirique exact (NASDAQ vs BATS, aout 2015, 108 titres, micro-prices, ~0.1 ms).